<a href="https://colab.research.google.com/github/Nurana100/flyrank-ml-starter-nurana/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nurana100/flyrank-ml-starter-nurana/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
*My lane is Refresh / Content Opportunity Scoring. The core task type is classification: predicting the probability that a page is declining (is_declining_label). That probability is then used to produce a ranking/scoring output — a queue of pages sorted by review priority. So the model itself is a classifier, but the product output is a ranked list, not a single yes/no answer.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
Target: is_declining_label = (trend_direction == "down"). This is a proxy, not a true future outcome — it's a bucket calculated from the current window, not something observed after a decision point. A stronger capstone version would use a future-looking label (e.g. features from the prior 90 days predicting decline over the next 30 days), but for this starter framing I'm using the beginner proxy the pipeline already ships with.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Nurana100/flyrank-ml-starter-nurana"
REPO_DIR = "flyrank-ml-starter-nurana"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

label_counts = df["trend_direction"].value_counts()
print(label_counts)
print(f"\nShare labeled 'down' (my proxy target): {(df['trend_direction']=='down').mean():.1%}")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Share labeled 'down' (my proxy target): 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*
Success metric: Precision@50. Given limited reviewer capacity, what matters is not overall accuracy but whether the top 50 pages the system flags are actually worth reviewing. Precision@50 directly answers "of the top 50 pages we'd tell a reviewer to look at first, how many are genuinely declining?" — which matches how the output would actually be used.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import subprocess
subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

import json
res = json.load(open("outputs/model_results.json"))
print(f"Baseline rule Precision@50: {res['baseline']['baseline_precision_at_50']:.3f}")
print(f"Random forest Precision@50: {res['models']['random_forest']['precision_at_50']:.3f}")


Baseline rule Precision@50: 0.240
Random forest Precision@50: 0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
One row = one content page (content_id), for one client (client_id), measured over a 90-day observation window. Below is a real slice of that dataframe showing the columns that matter for this lane: identifiers, visibility signals, staleness signals, and the trend label.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
        "avg_position", "ctr", "word_count", "trend_direction"]
df[cols].head(10)


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,word_count,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,3221.0,down
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,2481.0,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,3515.0,down
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,NaN,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,2803.0,down
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,3080.0,down
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,3059.0,down
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,NaN,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,0.09,3807.0,down
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,0.16,NaN,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A fixed rule (like "flag pages that are stale AND visible") only checks a couple of conditions at once. Real decline risk depends on how several signals interact together — staleness, visibility, position, CTR, word count — and the right combination isn't a single clean threshold. The starter pipeline shows this directly: the hand-written rule reaches Precision@50 of 0.240, while a random forest trained on the same signals reaches 0.740 — roughly a 3x improvement, verified using client-holdout validation (no client's pages appear in both train and test). That gap is evidence the useful pattern here is more complex than an if-statement can capture, even though the rule and the model start from the same information.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Same numbers as Section 3, shown again here to directly support this claim
print(f"Hand-written rule  Precision@50: {res['baseline']['baseline_precision_at_50']:.3f}")
print(f"Random forest      Precision@50: {res['models']['random_forest']['precision_at_50']:.3f}")
print(f"Lift: {res['models']['random_forest']['precision_at_50'] / res['baseline']['baseline_precision_at_50']:.1f}x")


Hand-written rule  Precision@50: 0.240
Random forest      Precision@50: 0.740
Lift: 3.1x


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.